# Übung 4
**Mustererkennung und Maschinelles Lernen, SS 2026. Frist: 24.06.2026**

Hinweise:
* Auf Google Colab mit GPU ausführen (Laufzeit > Laufzeittyp ändern > GPU).
* Stellen mit `# TODO` und `...` ergänzen Sie selbst. Lassen Sie vorgegebenen Code möglichst unverändert.

## Aufgabe 1: Feature Maps und Parameteranzahl

Bitte als Scan/Foto oder LaTeX hier einfügen.

# Aufgabe 2: Merkmalsextraktion und Finetuning

Mit einem auf ImageNet vortrainierten MobileNetV2 und wenigen Daten bauen wir einen Klassifikator für den Datensatz `tf_flowers` (5 Blumenklassen): zuerst per Merkmalsextraktion (eingefrorene Basis), dann per Fine-Tuning.

In [ ]:
# === Setup (vorgegeben) ===
import os
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
import tensorflow_datasets as tfds
from tensorflow import keras
from tensorflow.keras import layers

print("TensorFlow:", tf.__version__)
print("GPU verfügbar:", tf.config.list_physical_devices('GPU'))
assert tf.config.list_physical_devices('GPU'), "Keine GPU! Laufzeit > Laufzeittyp ändern > GPU"

SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

## a) Datensatz laden und aufteilen (vorgegeben)

Split: 65 % Training / 15 % Validierung / 20 % Test, Trainingsmenge reduziert auf 1500 Bilder.

In [ ]:
# === Daten laden und aufteilen (vorgegeben) ===
(ds_train_raw, ds_val_raw, ds_test_raw), info = tfds.load(
    'tf_flowers',
    split=['train[:65%]', 'train[65%:80%]', 'train[80%:]'],
    as_supervised=True,
    with_info=True,
)

NUM_CLASSES = info.features['label'].num_classes
class_names = info.features['label'].names
print("Klassen:", class_names)

N_TRAIN = 1500   # bewusst klein: Szenario "wenig Daten"
ds_train_raw = ds_train_raw.take(N_TRAIN)
print("Train:", sum(1 for _ in ds_train_raw),
      "| Val:", sum(1 for _ in ds_val_raw),
      "| Test:", sum(1 for _ in ds_test_raw))

plt.figure(figsize=(10, 6))
for i, (image, label) in enumerate(ds_train_raw.take(8)):
    plt.subplot(2, 4, i + 1)
    plt.imshow(image.numpy().astype('uint8'))
    plt.title(class_names[label.numpy()])
    plt.axis('off')
plt.tight_layout(); plt.show()

Warum wird hier neben einem Validierungsdatensatz noch ein Test-Set erzeugt?

In [ ]:
# === Preprocessing-Pipeline (vorgegeben) ===
# Skalieren auf 160x160, Wertebereich von MobileNetV2 ([-1, 1]), Batches, Prefetch.
IMG_SIZE = 160
BATCH_SIZE = 32

def preprocess(image, label):
    image = tf.image.resize(image, (IMG_SIZE, IMG_SIZE))
    image = keras.applications.mobilenet_v2.preprocess_input(image)
    return image, label

AUTOTUNE = tf.data.AUTOTUNE
ds_train = (ds_train_raw.map(preprocess, num_parallel_calls=AUTOTUNE)
            .cache().shuffle(1000, seed=SEED).batch(BATCH_SIZE).prefetch(AUTOTUNE))
ds_val   = ds_val_raw.map(preprocess, num_parallel_calls=AUTOTUNE).cache().batch(BATCH_SIZE).prefetch(AUTOTUNE)
ds_test  = ds_test_raw.map(preprocess, num_parallel_calls=AUTOTUNE).batch(BATCH_SIZE).prefetch(AUTOTUNE)

## b) Vortrainierte Basis laden und einfrieren

Laden Sie MobileNetV2 mit ImageNet-Gewichten ohne Klassifikationskopf und frieren Sie die
gesamte Faltungsbasis ein.

In [ ]:
base_model = keras.applications.MobileNetV2(
    input_shape=(IMG_SIZE, IMG_SIZE, 3),
    include_top=...,   # TODO
    weights=...,       # TODO
)

# TODO: Convolution-Basis einfrieren
...

print(f"Schichten in der Basis: {len(base_model.layers)}")
print(f"Parameter der Basis: {base_model.count_params():,}")

Wie viele Parameter hat die Basis, wie viele Trainingsbilder haben Sie? Wäre es sinnvoll, das gesamte Netz von Grund auf mit diesen Bildern zu trainieren?

## c) Neuen Klassifikator aufsetzen und trainieren (Merkmalsextraktion)

Setzen Sie auf die eingefrorene Basis: GlobalAveragePooling2D, Dropout(0.2), Dense mit passender Ausgabegrösse und Aktivierung. Trainiert wird nur dieser Kopf.

Welche Validierungs-Accuracy erwarten Sie nach 10 Epochen, in denen nur der Kopf trainiert wird: eher 30 %, 60 % oder 90 %? Kurze Begründung.

In [ ]:
inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 3))
x = base_model(inputs, training=False)   # nicht ändern (BatchNorm)

# TODO: GlobalAveragePooling2D, Dropout(0.2), Dense(?, activation=?)
x = ...
x = ...
outputs = ...

model = keras.Model(inputs, outputs)

# TODO: kompilieren. Welche Loss-Funktion passt zu ganzzahligen Labels (0..4) und Softmax?
model.compile(optimizer=keras.optimizers.Adam(learning_rate=1e-3),
              loss=...,   # TODO
              metrics=['accuracy'])
model.summary(show_trainable=True)

Wie viele trainierbare Parameter hat das Modell jetzt (siehe summary)?

Warum vermeidet GlobalAveragePooling das Flatten+Dense-Problem aus Aufgabe 1c?

In [ ]:
# === Training Phase 1 (vorgegeben) ===
EPOCHS_FE = 10
history_fe = model.fit(ds_train, validation_data=ds_val, epochs=EPOCHS_FE)

In [ ]:
# === Plot-Funktion (vorgegeben) ===
def plot_history(histories, title, vline=None):
    acc, val_acc, loss, val_loss = [], [], [], []
    for h in histories:
        acc += h.history['accuracy']; val_acc += h.history['val_accuracy']
        loss += h.history['loss'];    val_loss += h.history['val_loss']
    epochs = range(1, len(acc) + 1)

    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    axes[0].plot(epochs, acc, 'o-', label='Training')
    axes[0].plot(epochs, val_acc, 's-', label='Validierung')
    axes[0].set_xlabel('Epoche'); axes[0].set_ylabel('Accuracy'); axes[0].set_title('Genauigkeit')
    axes[1].plot(epochs, loss, 'o-', label='Training')
    axes[1].plot(epochs, val_loss, 's-', label='Validierung')
    axes[1].set_xlabel('Epoche'); axes[1].set_ylabel('Loss'); axes[1].set_title('Verlust')
    for ax in axes:
        if vline is not None:
            ax.axvline(vline + 0.5, color='gray', ls='--', label='Start Fine-Tuning')
        ax.legend(); ax.grid(alpha=0.3)
    fig.suptitle(title)
    plt.tight_layout(); plt.show()

plot_history([history_fe], 'Phase 1: Merkmalsextraktion (Basis eingefroren)')

Vergleichen Sie mit Ihrer Vorhersage. Wie nah liegen Trainings- und Validierungskurve beieinander, sehen Sie Overfitting? Warum (nicht)?

## d) Fine-Tuning der oberen Faltungsschichten

Geben Sie die oberen Schichten der Basis frei und trainieren Sie mit sehr kleiner Lernrate weiter.

Wählen Sie ein `FINE_TUNE_AT`, also ab welcher der 154 Schichten trainiert wird (sinnvoll: ca. 50 bis 140). Begründen Sie.

Was denken Sie, wie die Kurven in den Epochen 11 bis 20 weiterlaufen. Steigt, stagniert oder fällt die Validierungs-Accuracy?

Wichtig: Nach jeder Änderung von `trainable` muss neu kompiliert werden, sonst hat sie keinen Effekt.

In [ ]:
base_model.trainable = True

FINE_TUNE_AT = ...   # TODO (Freie Wahl, ca. 50 bis 140)
for layer in base_model.layers[:FINE_TUNE_AT]:
    layer.trainable = False

# TODO: neu kompilieren, mit deutlich kleinerer Lernrate (Empfehlung aus dem Blatt: 1e-5)
model.compile(optimizer=...,   # TODO
              loss=...,        # TODO (wie oben)
              metrics=['accuracy'])

n_trainable = int(np.sum([np.prod(v.shape) for v in model.trainable_weights]))
print(f"Trainierbare Parameter jetzt: {n_trainable:,}")

EPOCHS_FT = 10
history_ft = model.fit(ds_train, validation_data=ds_val,
                       epochs=EPOCHS_FE + EPOCHS_FT,
                       initial_epoch=EPOCHS_FE)

In [ ]:
plot_history([history_fe, history_ft],
             'Merkmalsextraktion (Epochen 1-10) + Fine-Tuning (Epochen 11-20)',
             vline=EPOCHS_FE)

test_loss, test_acc = model.evaluate(ds_test, verbose=0)
print(f"Test-Accuracy nach Fine-Tuning: {test_acc:.3f}")

## e) Diskussion

Warum trainiert man beim Fine-Tuning mit einer so kleinen Lernrate?

Warum gibt man nur die oberen Schichten frei und nicht die unteren?

# Aufgabe 3: Embeddings als Repräsentationen

Ein Feature-Extractor bildet jedes Bild auf einen Vektor (Embedding) ab. Wir prüfen, ob diese
Embeddings semantisch sinnvoll sind: Bleiben sich ähnliche Ziffern auch im Embedding-Raum nahe?

Als Extractor dient ein LeNet-5, das fertig vorgegeben ist und auf MNIST schnell trainiert ist. Ihre Aufgabe beginnt danach: Embeddings extrahieren und untersuchen.

In [1]:
# === MNIST laden (vorgegeben) ===
(x_train, y_train), (x_test, y_test) = keras.datasets.mnist.load_data()
x_train_norm = x_train[..., None].astype('float32') / 255.0
x_test_norm  = x_test[..., None].astype('float32') / 255.0
print("Trainingsdaten:", x_train_norm.shape)

NameError: name 'keras' is not defined

In [ ]:
# === Feature-Extractor: LeNet-5 trainieren (vorgegeben, nur ausführen) ===
# LeNet-5 (LeCun et al., 1998) ist eine der ersten erfolgreichen CNN-Architekturen
# und wurde für genau diese Aufgabe entworfen: handgeschriebene Ziffern.
# Wir bleiben nahe am Original (tanh-Aktivierung, Average Pooling); moderne Netze
# verwenden stattdessen meist ReLU und Max Pooling.

inp = keras.Input(shape=(28, 28, 1))
h = layers.Conv2D(6, 5, padding='same', activation='tanh')(inp)   # C1
h = layers.AveragePooling2D(2)(h)                                 # S2
h = layers.Conv2D(16, 5, activation='tanh')(h)                    # C3
h = layers.AveragePooling2D(2)(h)                                 # S4
h = layers.Flatten()(h)
h = layers.Dense(120, activation='tanh')(h)                       # C5
h = layers.Dense(84, activation='tanh', name='embedding')(h)      # F6
out = layers.Dense(10, activation='softmax')(h)
lenet = keras.Model(inp, out, name='LeNet5')

lenet.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
lenet.fit(x_train_norm, y_train, epochs=5, batch_size=128, validation_split=0.1, verbose=2)
print('Test-Accuracy:', lenet.evaluate(x_test_norm, y_test, verbose=0)[1])

## a) Balancierte Teilmenge ziehen und normalisieren

Schreiben Sie `balanced_subset`: pro Ziffer genau `n_per_class` zufällige Bilder ohne Zurücklegen,
gemischt zurückgeben. Danach Pixel auf [0, 1] normalisieren.
Bausteine: `np.where(y == c)[0]`, `rng.choice(..., replace=False)`, `np.concatenate`, `rng.shuffle`.

In [ ]:
def balanced_subset(x, y, n_per_class, seed=SEED):
    rng = np.random.default_rng(seed)

    # Wir ziehen fuer jede Ziffer 0 bis 9 genau gleich viele Beispiele.
    # Dadurch ist die Teilmenge balanciert: keine Ziffer kommt haeufiger vor
    # und beeinflusst t-SNE oder K-Means staerker als die anderen.
    all_indices = []
    for c in range(10):
        indices_c = np.where(y == c)[0]
        chosen_c = rng.choice(indices_c, size=n_per_class, replace=False)
        all_indices.append(chosen_c)

    # Aus den zehn kleinen Index-Listen wird eine grosse Liste gebaut.
    # Danach mischen wir, damit die Bilder nicht nach Klassen sortiert sind.
    idx = np.concatenate(all_indices)
    rng.shuffle(idx)

    return x[idx], y[idx]

N_PER_CLASS = 250   # 2500 Bilder, fuer Vergleichbarkeit nicht aendern
x_sub, y_sub = balanced_subset(x_train, y_train, N_PER_CLASS)

# Die Rohdaten liegen als Grauwerte von 0 bis 255 vor.
# Fuer das trainierte Netz brauchen wir wieder Werte im Bereich [0, 1].
x_sub = x_sub.astype('float32') / 255.0

# Selbsttest (vorgegeben), muss ohne Fehler durchlaufen:
assert x_sub.shape == (2500, 28, 28) and x_sub.max() <= 1.0
assert np.all(np.bincount(y_sub) == N_PER_CLASS), "Teilmenge ist nicht klassenbalanciert!"
print("OK. Teilmenge:", x_sub.shape, "| Klassenverteilung:", np.bincount(y_sub))


## b) Extractor bauen und Embeddings berechnen

Erstellen Sie aus dem trainierten LeNet ein Teilmodell bis zur Schicht `embedding` und berechnen
Sie mit einem `predict`-Aufruf für jedes Bild ein Embedding.
Hinweis: `lenet.get_layer('embedding').output` liefert den Ausgang der Schicht; daraus lässt sich
mit `keras.Model(...)` ein Teilmodell bauen. Eingabeform: `x_sub[..., None]`.

In [ ]:
# Das fertige LeNet endet mit einer Softmax-Ausgabe fuer die 10 Ziffern.
# Fuer Embeddings wollen wir aber nicht die finale Klasse, sondern die 84-dimensionalen
# Merkmale aus der Schicht mit dem Namen "embedding".
extractor = keras.Model(
    inputs=lenet.input,
    outputs=lenet.get_layer('embedding').output
)

# x_sub hat noch die Form (2500, 28, 28).
# Das CNN erwartet aber (2500, 28, 28, 1), also ergaenzen wir den Kanal mit [..., None].
# Danach liefert predict fuer jedes Bild einen 84-dimensionalen Embedding-Vektor.
embeddings = extractor.predict(x_sub[..., None], batch_size=128, verbose=0)

print('Form der Embeddings:', embeddings.shape)


Welche Dimension hat ein einzelnes Embedding? Um welchen Faktor wurde die
Darstellung gegenüber den Rohpixeln (28x28 = 784 Werte) reduziert?

## c) Baseline: t-SNE auf Rohpixeln

Wenden Sie t-SNE direkt auf die flachgemachten Rohpixel an (jedes Bild als 784-dim Vektor) und
plotten Sie das Ergebnis eingefärbt nach der wahren Ziffer.

Vorgeschlagene Parameter: `perplexity=30, init='pca', learning_rate='auto',
random_state=SEED`.

In [ ]:
from sklearn.manifold import TSNE

# === Plot-Funktion (vorgegeben) ===
def plot_tsne(points_2d, labels, title, ax=None):
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 6))
    sc = ax.scatter(points_2d[:, 0], points_2d[:, 1], c=labels, cmap='tab10', s=6, alpha=0.8)
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([])
    return sc

# TODO: Bilder zu (2500, 784) flach machen
x_flat = ...

# TODO: t-SNE auf den Rohpixeln (Parameter siehe oben)
z_raw = ...

fig, ax = plt.subplots(figsize=(7, 6))
sc = plot_tsne(z_raw, y_sub, 't-SNE auf Rohpixeln (784-dim)', ax)
fig.colorbar(sc, ax=ax, ticks=range(10), label='Ziffer')
plt.show()

## d) t-SNE auf den Embeddings und Vergleich

Vor dem Ausführen: Erwarten Sie auf den Embeddings klarere oder unklarere Cluster als auf den Rohpixeln? Warum? Welche Ziffernpaare könnten in beiden Fällen schwer zu trennen sein?

In [ ]:
# TODO: t-SNE auf den Embeddings (gleiche Parameter wie in c)
z_emb = ...

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_tsne(z_raw, y_sub, 'Rohpixel (784-dim)', axes[0])
sc = plot_tsne(z_emb, y_sub, f'Embeddings ({embeddings.shape[1]}-dim)', axes[1])
fig.colorbar(sc, ax=axes, ticks=range(10), label='Ziffer', shrink=0.8)
plt.show()

In welchem Fall bilden die Ziffern klarere, besser getrennte Cluster?

Was schliessen Sie daraus über die Qualität der gelernten Repräsentation? (Was bedeutet hier "Ähnlichkeit": Bild-Ähnlichkeit oder Bedeutungs-Ähnlichkeit?)

## e) Cluster quantifizieren: K-Means und Adjusted Rand Index

Wenden Sie K-Means mit 10 Clustern (`n_init=10, random_state=SEED`) auf die Embeddings an und
messen Sie die Übereinstimmung mit den wahren Labels per Adjusted Rand Index. Berechnen Sie zum
Vergleich denselben Wert für K-Means auf den Rohpixeln.

In [ ]:
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

# TODO: K-Means (n_clusters=10, n_init=10, random_state=SEED) auf Embeddings und Rohpixeln
km_emb = ...
km_raw = ...

# TODO: ARI für beide Fälle
ari_emb = ...
ari_raw = ...
print(f'ARI auf Embeddings: {ari_emb:.3f}')
print(f'ARI auf Rohpixeln : {ari_raw:.3f}')

In [ ]:
# === Kontingenztabelle (vorgegeben) ===
import pandas as pd
cont = pd.crosstab(pd.Series(y_sub, name='Ziffer'), pd.Series(km_emb.labels_, name='Cluster'))

plt.figure(figsize=(8, 6))
plt.imshow(cont, cmap='Blues')
plt.colorbar(label='Anzahl Bilder')
plt.xlabel('K-Means-Cluster'); plt.ylabel('Wahre Ziffer')
plt.xticks(range(10)); plt.yticks(range(10))
for i in range(10):
    for j in range(10):
        v = cont.iloc[i, j]
        if v > 0:
            plt.text(j, i, v, ha='center', va='center',
                     color='white' if v > cont.values.max()/2 else 'black', fontsize=8)
plt.title('Wahre Ziffer vs. K-Means-Cluster (Embeddings)')
plt.tight_layout(); plt.show()

purity = (cont.max(axis=1) / cont.sum(axis=1)).sort_values()
print('Anteil der Bilder im jeweils grössten Cluster (niedrig = oft verwechselt):')
print(purity.round(2))

Vergleichen Sie die beiden ARI-Werte. Was sagt der Unterschied über die Embeddings aus? Welche Ziffern werden tatsächlich verwechselt, deckt sich das mit Ihrer Vorhersage?